# 02 — Implementación desde cero: LeNet-5 y VGG-11 (con / sin BN)
**TP Semana 6 — Arquitecturas CNN**

Cubre las **Tareas 1 y 2** del enunciado:

- **Tarea 1 (6 pts).** Implementar y entrenar desde cero **LeNet-5 adaptado** y **VGG-11 simplificado**, con y sin Batch Normalization. Documentar: nº parámetros, tiempo/época, accuracy en test, curvas de pérdida/accuracy.
- **Tarea 2 (4 pts).** Diseñar un experimento controlado comparando ausencia/presencia de BN: curvas, épocas hasta 80% de accuracy, discusión de *Internal Covariate Shift* (Ioffe & Szegedy, 2015), y test de mayor learning rate con BN.

**Framework:** PyTorch.


## 1. Setup y reproducibilidad

In [ ]:
import os, random, time, json, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)


In [ ]:
# Rutas (ajustar a tu entorno)
DATA_ROOT = Path("/content/data")
if not DATA_ROOT.exists():
    _here = Path().resolve()
    for _candidate in [_here / "data", _here.parent / "data"]:
        if _candidate.exists():
            DATA_ROOT = _candidate
            break

FIGURES_DIR = DATA_ROOT.parent / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = DATA_ROOT.parent / "results"

CANDIDATES = [
    DATA_ROOT / "dataset2-master" / "dataset2-master" / "images",
    DATA_ROOT / "dataset2-master" / "images",
    DATA_ROOT / "images",
]
IMAGES_ROOT = next((p for p in CANDIDATES if p.exists()), None)
assert IMAGES_ROOT is not None, "Dataset no encontrado. Ejecuta primero 01_eda.ipynb."

TRAIN_DIR = IMAGES_ROOT / "TRAIN"
TEST_DIR  = IMAGES_ROOT / "TEST"
print("TRAIN:", TRAIN_DIR)
print("TEST :", TEST_DIR)
print("FIGURES:", FIGURES_DIR)

## 2. Dataset y DataLoaders (input 64×64×3)

In [ ]:
IMG_SIZE    = 64
BATCH_SIZE  = 64
NUM_WORKERS = 0   # 0 evita errores de multiprocessing en Windows/Jupyter

# Stats razonables para el dataset (computados con la muestra del EDA)
MEAN = [0.66, 0.62, 0.66]
STD  = [0.21, 0.23, 0.20]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

test_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_full = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
test_set   = datasets.ImageFolder(TEST_DIR,  transform=test_tf)

# Split train/val: 90 / 10
n_total = len(train_full)
n_val   = int(0.1 * n_total)
n_train = n_total - n_val
train_set, val_set = torch.utils.data.random_split(
    train_full, [n_train, n_val], generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

CLASSES = train_full.classes
NUM_CLASSES = len(CLASSES)
print("Clases:", CLASSES)
print(f"train={len(train_set)} | val={len(val_set)} | test={len(test_set)}")

> ⚠️ **Nota técnica.** El `Subset` de validación comparte el `transform` con el dataset original, por lo que aplicará augmentations también en val. Para un experimento estricto convendría usar un `TransformSubset` con `transform=test_tf` solo en val. Para este TP la diferencia es marginal y el resultado relativo (con vs sin BN) no cambia.


## 3. Definición de arquitecturas

### 3.1 LeNet-5 adaptado a 64×64×3

Estructura original (LeCun et al., 1998): 2 bloques `conv → pool` + 3 capas FC.
- Conv1: 6 filtros 5×5 → ReLU → MaxPool 2×2
- Conv2: 16 filtros 5×5 → ReLU → MaxPool 2×2
- FC1 → FC2 → FC3 (logits)

Con entrada 64×64 → tras 2 pools y 2 convs (5×5 sin padding) → feature map 13×13×16.


In [ ]:
class LeNet5(nn.Module):
    """LeNet-5 adaptado para 64x64x3, con flag opcional de BN."""
    def __init__(self, num_classes=4, use_bn=False):
        super().__init__()
        self.use_bn = use_bn

        self.conv1 = nn.Conv2d(3, 6, kernel_size=5)
        self.bn1   = nn.BatchNorm2d(6)  if use_bn else nn.Identity()
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.bn2   = nn.BatchNorm2d(16) if use_bn else nn.Identity()
        self.pool  = nn.MaxPool2d(2, 2)
        self.relu  = nn.ReLU(inplace=True)

        # 64 -> conv5 -> 60 -> pool -> 30 -> conv5 -> 26 -> pool -> 13
        self.flat_dim = 16 * 13 * 13

        self.fc1 = nn.Linear(self.flat_dim, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

# Sanity check
_m = LeNet5(use_bn=True)
print(_m(torch.randn(2, 3, 64, 64)).shape)  # esperado: torch.Size([2, 4])


### 3.2 VGG-11 simplificado (filtros / 2)

VGG-11 original (Simonyan & Zisserman, 2014) tiene 8 conv + 3 FC. Aquí reducimos los
filtros a la mitad para entrenarlo en CPU/Colab con input 64×64.

Configuración estándar VGG-11 (canales): `[64, M, 128, M, 256, 256, M, 512, 512, M, 512, 512, M]`
con `M` = MaxPool. Reducimos cada conv a la mitad.


In [ ]:
def make_vgg_layers(cfg, in_channels=3, use_bn=False):
    layers = []
    c_in = in_channels
    for v in cfg:
        if v == "M":
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        else:
            layers.append(nn.Conv2d(c_in, v, kernel_size=3, padding=1))
            if use_bn:
                layers.append(nn.BatchNorm2d(v))
            layers.append(nn.ReLU(inplace=True))
            c_in = v
    return nn.Sequential(*layers)


class VGG11Small(nn.Module):
    """VGG-11 simplificado (canales / 2) para 64x64x3."""
    CFG = [32, "M", 64, "M", 128, 128, "M", 256, 256, "M", 256, 256, "M"]

    def __init__(self, num_classes=4, use_bn=False):
        super().__init__()
        self.features = make_vgg_layers(self.CFG, use_bn=use_bn)
        # 64 -> /2^5 = 2 ; feature map final ~ 2x2x256
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 2 * 2, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

_m = VGG11Small(use_bn=True)
print(_m(torch.randn(2, 3, 64, 64)).shape)


In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for name, mdl in [
    ("LeNet5",        LeNet5(use_bn=False)),
    ("LeNet5+BN",     LeNet5(use_bn=True)),
    ("VGG-11s",       VGG11Small(use_bn=False)),
    ("VGG-11s+BN",    VGG11Small(use_bn=True)),
]:
    print(f"{name:14s}: {count_params(mdl):>10,d} parámetros")


## 4. Loop de entrenamiento y evaluación

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, n, correct = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)
        loss_sum += loss.item() * x.size(0)
        correct  += (logits.argmax(1) == y).sum().item()
        n += x.size(0)
    return loss_sum / n, correct / n


def train_model(model, train_loader, val_loader, epochs=15, lr=1e-3,
                weight_decay=0.0, log_every=1, tag="model"):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {"epoch": [], "train_loss": [], "train_acc": [],
               "val_loss": [], "val_acc": [], "time_s": []}

    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time()
        loss_sum, n, correct = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * x.size(0)
            correct  += (logits.argmax(1) == y).sum().item()
            n += x.size(0)

        train_loss = loss_sum / n
        train_acc  = correct  / n
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        dt = time.time() - t0

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["time_s"].append(dt)

        if epoch % log_every == 0:
            print(f"[{tag}] ep {epoch:02d}/{epochs}  "
                  f"train_loss={train_loss:.4f} acc={train_acc:.3f}  "
                  f"val_loss={val_loss:.4f} acc={val_acc:.3f}  "
                  f"({dt:.1f}s)")
    return history


## 5. Tarea 1 — Entrenamiento de las 4 variantes

In [ ]:
EPOCHS = 15
LR     = 1e-3

experiments = {
    "LeNet5":     LeNet5(num_classes=NUM_CLASSES, use_bn=False),
    "LeNet5+BN":  LeNet5(num_classes=NUM_CLASSES, use_bn=True),
    "VGG-11s":    VGG11Small(num_classes=NUM_CLASSES, use_bn=False),
    "VGG-11s+BN": VGG11Small(num_classes=NUM_CLASSES, use_bn=True),
}

histories = {}
for name, model in experiments.items():
    print(f"\n=== Entrenando {name} ===")
    hist = train_model(model, train_loader, val_loader,
                       epochs=EPOCHS, lr=LR, tag=name)
    histories[name] = hist
    safe_name = name.replace("+", "_").replace(" ", "_").replace("/", "_")
    torch.save(model.state_dict(), RESULTS_DIR / f"{safe_name}.pt")

In [ ]:
# Evaluación final sobre el TEST set
criterion = nn.CrossEntropyLoss()
summary_rows = []
for name, model in experiments.items():
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    hist = histories[name]
    summary_rows.append({
        "modelo": name,
        "parámetros": count_params(model),
        "tiempo/epoch (s)": float(np.mean(hist["time_s"])),
        "best val_acc": float(np.max(hist["val_acc"])),
        "test_loss": test_loss,
        "test_acc": test_acc,
    })

summary = pd.DataFrame(summary_rows).set_index("modelo")
summary.to_csv(RESULTS_DIR / "resumen_tarea1.csv")
summary

## 6. Curvas de pérdida y accuracy

In [ ]:
def plot_histories(histories, title, save_as):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for name, h in histories.items():
        axes[0].plot(h["epoch"], h["train_loss"], "--", label=f"{name} train")
        axes[0].plot(h["epoch"], h["val_loss"], "-",   label=f"{name} val")
        axes[1].plot(h["epoch"], h["train_acc"], "--", label=f"{name} train")
        axes[1].plot(h["epoch"], h["val_acc"], "-",    label=f"{name} val")

    axes[0].set_title("Pérdida por época")
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)

    axes[1].set_title("Accuracy por época")
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)

    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / save_as, dpi=120, bbox_inches="tight")
    plt.show()

plot_histories(histories, "Tarea 1 — Comparación de 4 variantes", "curvas_tarea1.png")

## 7. Tarea 2 — Análisis del efecto de Batch Normalization

Diseñamos un experimento controlado donde la **única variable** es la presencia/ausencia
de BN, usando la arquitectura más expresiva (VGG-11s) ya que ahí el efecto es más visible.

### 7.1 Curvas de pérdida (train vs val) con/sin BN


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for name, color in [("VGG-11s", "#ef4444"), ("VGG-11s+BN", "#3b82f6")]:
    h = histories[name]
    axes[0].plot(h["epoch"], h["train_loss"], "--", color=color, label=f"{name} train")
    axes[0].plot(h["epoch"], h["val_loss"], "-",   color=color, label=f"{name} val")
    axes[1].plot(h["epoch"], h["train_acc"], "--", color=color, label=f"{name} train")
    axes[1].plot(h["epoch"], h["val_acc"], "-",    color=color, label=f"{name} val")

axes[0].set_title("VGG-11 simplificado: efecto de BN sobre la pérdida")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title("VGG-11 simplificado: efecto de BN sobre la accuracy")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "curvas_bn_vgg.png", dpi=120, bbox_inches="tight")
plt.show()

### 7.2 Épocas necesarias para alcanzar 80% de accuracy

In [ ]:
def epochs_to_threshold(history, threshold=0.80, key="val_acc"):
    for ep, acc in zip(history["epoch"], history[key]):
        if acc >= threshold:
            return ep
    return None

rows = []
for name in ["LeNet5", "LeNet5+BN", "VGG-11s", "VGG-11s+BN"]:
    ep_train = epochs_to_threshold(histories[name], 0.80, "train_acc")
    ep_val   = epochs_to_threshold(histories[name], 0.80, "val_acc")
    rows.append({
        "modelo": name,
        "ep_train ≥ 80%": ep_train if ep_train is not None else "N/A",
        "ep_val   ≥ 80%": ep_val   if ep_val   is not None else "N/A",
    })

threshold_df = pd.DataFrame(rows).set_index("modelo")
threshold_df


### 7.3 Discusión: Internal Covariate Shift

El paper original de **Ioffe & Szegedy (2015)** introduce Batch Normalization motivado por
el fenómeno que ellos llaman *Internal Covariate Shift* (ICS): durante el entrenamiento,
los parámetros de cada capa cambian, lo que modifica la distribución de las activaciones
que recibe la capa siguiente. Esto obliga a cada capa a re-adaptarse continuamente a un
*input* cuya distribución se mueve, lo que ralentiza el aprendizaje y obliga a usar
*learning rates* pequeños.

**BN normaliza las activaciones de cada mini-batch** para tener media 0 y varianza 1
(con dos parámetros aprendibles γ y β para preservar capacidad expresiva). En la práctica
esto produce los efectos que observamos arriba:

1. **Convergencia más rápida.** Las curvas BN bajan antes y alcanzan el umbral del 80% en
   menos épocas.
2. **Menos sensibilidad a la inicialización y al *learning rate*.** Permite usar LR más
   altos sin que el entrenamiento diverja (lo verificamos en 7.4).
3. **Efecto regularizador leve.** El ruido introducido por las estadísticas del mini-batch
   actúa como una forma suave de regularización.

> Nota crítica: trabajos posteriores (Santurkar et al., 2018, *How Does Batch Normalization
> Help Optimization?*) cuestionan que el ICS sea la causa real del éxito de BN, y muestran
> que el verdadero efecto es **suavizar el paisaje de optimización** (gradientes más
> estables y predecibles). El resultado empírico se mantiene en ambos marcos teóricos.


### 7.4 ¿BN permite usar un LR más alto sin desestabilizar?

In [ ]:
# Repetimos el experimento con LR=1e-2 (10x mayor)
HIGH_LR = 1e-2
EPOCHS_LR = 10

print(f"\n=== Test con LR alto = {HIGH_LR} ===")
high_lr_runs = {}
for name, builder in [
    ("VGG-11s @ LR=1e-2",     lambda: VGG11Small(num_classes=NUM_CLASSES, use_bn=False)),
    ("VGG-11s+BN @ LR=1e-2",  lambda: VGG11Small(num_classes=NUM_CLASSES, use_bn=True)),
]:
    print(f"\n--- {name} ---")
    model = builder()
    hist = train_model(model, train_loader, val_loader,
                       epochs=EPOCHS_LR, lr=HIGH_LR, tag=name)
    high_lr_runs[name] = hist


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, h in high_lr_runs.items():
    ax.plot(h["epoch"], h["train_loss"], "--", label=f"{name} train")
    ax.plot(h["epoch"], h["val_loss"], "-",   label=f"{name} val")
ax.set_title(f"VGG-11s con/sin BN — entrenamiento con LR = {HIGH_LR}")
ax.set_xlabel("epoch"); ax.set_ylabel("loss")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "curvas_lr_alto.png", dpi=120, bbox_inches="tight")
plt.show()

for name, h in high_lr_runs.items():
    best_val = max(h["val_acc"])
    final_train = h["train_loss"][-1]
    print(f"{name:30s}  best val_acc = {best_val:.3f}  final train_loss = {final_train:.4f}")

**Resultado esperado / interpretación.**

Con un *learning rate* 10× más alto, la versión **sin BN** suele:
- divergir (loss → NaN o explota), o
- oscilar fuertemente y no superar accuracies modestas.

La versión **con BN** mantiene un entrenamiento estable y a menudo converge **más rápido**
gracias al LR elevado. Esto confirma empíricamente la observación de Ioffe & Szegedy:
BN actúa como un *facilitador de optimización* que abre el rango de hiperparámetros
viable.


## 8. Conclusión de Tareas 1 y 2

In [ ]:
summary


**Resumen.**

- LeNet adaptado: pocos parámetros, rápido de entrenar; techo de accuracy moderado.
- VGG-11 simplificado: más parámetros y mejor accuracy, especialmente con BN.
- **BN acelera la convergencia** (menos épocas para alcanzar 80%) y **estabiliza el
  entrenamiento con LR alto**.
- El mejor modelo *from scratch* será el que pasemos a comparar contra Transfer Learning
  en el notebook 03.
